# CNN, paper-aligned
Architecture from the paper's CNN branch: 3 conv layers (kernels 3/5/7, filters 128/128/256), ReLU, max pooling. The paper doesn't say whether the layers are stacked or parallel; they are **stacked** here.

Training (paper Sec. 4.4): batch 32, up to 15 epochs, early stopping patience 3, seeds 42/123/2025 (mean ± std), 70/15/15 stratified split from `train/val/test.jsonl`. Optimizer is Adam with lr 1e-3 (not the paper's 2e-5, which is for fine-tuning XLM-R). Input is raw Gujarati text with a trained-from-scratch embedding (the paper's full model instead feeds XLM-R embeddings into these branches).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, gc, numpy as np, pandas as pd

# ---- EDIT: Drive folder that holds gujarati_fake_real_combined.jsonl and the split files ----
DATA_DIR = "/content/drive/MyDrive/RM_24DCS041_054"
MODEL_DIR = f"{DATA_DIR}/models_paper"      # separate folder so earlier models are not overwritten
os.makedirs(MODEL_DIR, exist_ok=True)

SEEDS = [42, 123, 2025]   # paper (Sec. 4.4/5.1): three seeds, results reported as mean +/- std

In [ ]:
def load_split(name):
    d = pd.read_json(f"{DATA_DIR}/{name}.jsonl", lines=True, dtype={"news_id": str})
    assert {"news_id", "text", "label"} <= set(d.columns)
    d["text"] = d["text"].astype(str)
    d["label"] = d["label"].astype(str).str.strip().str.lower()
    assert d["label"].isin(["fake", "real"]).all()
    return d

train_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")

# one label encoding for every model: real = 0, fake = 1
label2id = {"real": 0, "fake": 1}
id2label = {0: "real", 1: "fake"}

X_train, y_train = train_df["text"].values, train_df["label"].map(label2id).values
X_val,   y_val   = val_df["text"].values,   val_df["label"].map(label2id).values
X_test,  y_test  = test_df["text"].values,  test_df["label"].map(label2id).values

n = len(train_df) + len(val_df) + len(test_df)
print(f"train {len(train_df)} ({len(train_df)/n:.1%}) | val {len(val_df)} ({len(val_df)/n:.1%}) | test {len(test_df)} ({len(test_df)/n:.1%})")
print("Expected ~70/15/15. If not, re-run 01_data_split.ipynb first.")
print(train_df["label"].value_counts().to_dict())

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

def evaluate(y_true, y_pred, n_boot=1000, seed=0):
    """Macro metrics + bootstrap 95% CI for macro-F1 (paper Sec. 5.1)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        boots.append(f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_precision": p,
            "macro_recall": r, "macro_f1": f, "f1_ci_low": lo, "f1_ci_high": hi}

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# paper Sec. 4.4: batch 32, up to 15 epochs, early stopping patience 3
BATCH_SIZE, EPOCHS, PATIENCE = 32, 15, 3
MAX_TOKENS, SEQ_LEN, EMB_DIM = 50000, 300, 128
LR = 1e-3   # Adam; the paper's 2e-5 is for fine-tuning XLM-R, not for from-scratch networks

def make_vectorizer():
    v = layers.TextVectorization(max_tokens=MAX_TOKENS, output_mode="int", output_sequence_length=SEQ_LEN,
                                 standardize=None, split="whitespace")
    v.adapt(X_train)
    return v

def make_ds(X, y, shuffle=False, seed=0):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(len(X), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds, test_ds = make_ds(X_val, y_val), make_ds(X_test, y_test)

In [ ]:
def build_model(vectorizer):
    """Paper Sec. 4.1 CNN branch: 3 conv layers, kernels {3,5,7}, filters {128,128,256}, ReLU + max pooling."""
    vocab_size = len(vectorizer.get_vocabulary())
    inp = keras.Input(shape=(1,), dtype=tf.string, name="raw_text")
    x = vectorizer(inp)
    x = layers.Embedding(vocab_size, EMB_DIM, name="embedding")(x)
    x = layers.Conv1D(128, 3, activation="relu", padding="same", name="conv_k3")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 5, activation="relu", padding="same", name="conv_k5")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(256, 7, activation="relu", padding="same", name="conv_k7")(x)
    x = layers.GlobalMaxPooling1D(name="global_max_pool")(x)
    x = layers.Dropout(0.4)(x)                      # paper classifier: dropout 0.4, hidden 128
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation="sigmoid", name="classifier")(x)
    return keras.Model(inp, out, name="Gujarati_CNN")

In [ ]:
results, all_preds, histories = [], {}, {}

for seed in SEEDS:
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    vectorizer = make_vectorizer()
    model = build_model(vectorizer)
    if seed == SEEDS[0]:
        model.summary()
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss="binary_crossentropy", metrics=["accuracy"])

    hist = model.fit(
        make_ds(X_train, y_train, shuffle=True, seed=seed), validation_data=val_ds,
        epochs=EPOCHS, verbose=1,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=PATIENCE,
                                                 restore_best_weights=True, verbose=1)])

    val_pred = (model.predict(val_ds, verbose=0).ravel() >= 0.5).astype(int)
    pred = (model.predict(test_ds, verbose=0).ravel() >= 0.5).astype(int)
    m = evaluate(y_test, pred)
    m.update(seed=seed, val_macro_f1=f1_score(y_val, val_pred, average="macro"))
    results.append(m); all_preds[seed] = pred; histories[seed] = hist.history
    print(f"seed {seed}: val macro-F1 {m['val_macro_f1']:.4f} | test macro-F1 {m['macro_f1']:.4f}")
    model.save(f"{MODEL_DIR}/10_cnn_seed{seed}.keras")

In [ ]:
h = histories[SEEDS[0]]
ep = range(1, len(h["accuracy"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(ep, h["accuracy"], "o-", label="train"); a1.plot(ep, h["val_accuracy"], "s-", label="val"); a1.set_title("Accuracy"); a1.legend()
a2.plot(ep, h["loss"], "o-", label="train"); a2.plot(ep, h["val_loss"], "s-", label="val"); a2.set_title("Loss"); a2.legend()
for a in (a1, a2): a.set_xlabel("epoch"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
res = pd.DataFrame(results).set_index("seed")
display(res.round(4))
cols = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
summ = res[cols].agg(["mean", "std"]).T
print("=== TEST RESULTS, mean +/- std over seeds " + str(list(res.index)) + " ===")
for c in cols:
    print(f"{c:16s}: {summ.loc[c, 'mean']:.4f} +/- {summ.loc[c, 'std']:.4f}")
res.to_csv(f"{MODEL_DIR}/10_cnn_results.csv")

first = SEEDS[0]
print(f"\nClassification report (seed {first}):")
print(classification_report(y_test, all_preds[first], target_names=["real", "fake"], digits=4, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, all_preds[first], display_labels=["real", "fake"], cmap="Blues")
plt.title(f"10_cnn - confusion matrix (seed {first})"); plt.show()